In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from pathlib import Path
import os

import pyrootutils

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")
os.environ.setdefault("TF_XLA_FLAGS", "--tf_xla_auto_jit=0")
os.environ.setdefault("TF_GPU_ALLOCATOR", "cuda_malloc_async")

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator="pyproject.toml",
    pythonpath=True,
    dotenv=True,
)



In [4]:
from building.scaling import (
    ScalingRunConfig,
    load_full_arrays,
    run_experiments,
    summarize_results,
    plot_summary,
)

2026-04-10 09:15:19.697987: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-10 09:15:19.712004: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-10 09:15:19.712035: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [ ]:
COLLECTION = "diff_family"
N_SAMPLES = 20
EPOCHS = 30
PATIENCE = 3
BATCH_SIZE = 32
SEED = 42
THRESHOLD = 0.5
BUILD_MODEL = "cnn2d"
INPUT_REPR = "mel"
MODELS_DIR = ROOT / "models" / COLLECTION / ("scaling_"+INPUT_REPR)
RESULTS_FILE = MODELS_DIR / "results.jsonl"

In [6]:
config = ScalingRunConfig(
    collection=COLLECTION,
    build_model=BUILD_MODEL,
    epochs=EPOCHS,
    patience=PATIENCE,
    batch_size=BATCH_SIZE,
    seed=SEED,
    threshold=THRESHOLD,
    models_dir=MODELS_DIR,
    results_file=RESULTS_FILE,
    input_repr=INPUT_REPR,
)

arrays = load_full_arrays(
    collection=COLLECTION,
    batch_size=BATCH_SIZE,
    seed=SEED,
    input_repr=INPUT_REPR,
)

print(f"Loaded {len(arrays.class_names)} classes:")
print(arrays.class_names)

Found 9016 files belonging to 11 classes.
Found 1954 files belonging to 11 classes.
Found 1958 files belonging to 11 classes.
Loaded 11 classes:
['Alauda_arvensis', 'Corvus_cornix', 'Emberiza_citrinella', 'Erithacus_rubecula', 'Fringilla_coelebs', 'Parus_major', 'Phylloscopus_collybita', 'Sylvia_atricapilla', 'Troglodytes_troglodytes', 'Turdus_merula', 'non_target']


In [7]:
baseline_rows = run_experiments(arrays, config, run_baseline=True, run_scaling=False)
print(f"New baseline runs: {len(baseline_rows)}")
if baseline_rows:
    print("Last baseline row:")
    print(baseline_rows[-1])

[baseline] target=Alauda_arvensis
[baseline] n_each=339 n_non_target=339 n_classes=2
Epoch 1/30


I0000 00:00:1775805391.697599   11506 service.cc:145] XLA service 0x76652400b650 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775805391.697623   11506 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
I0000 00:00:1775805393.502467   11506 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
I0000 00:00:1775805395.165094   11505 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'input_reduce_select_fusion_1', 68 bytes spill stores, 68 bytes spill loads



22/22 - 5s - 228ms/step - accuracy: 0.4956 - loss: 0.7166 - precision: 0.4969 - recall: 0.4705 - val_accuracy: 0.2021 - val_loss: 0.6938 - val_precision: 0.2021 - val_recall: 0.2021
Epoch 2/30
22/22 - 0s - 6ms/step - accuracy: 0.5000 - loss: 0.6932 - precision: 0.5000 - recall: 0.5000 - val_accuracy: 0.2021 - val_loss: 0.6935 - val_precision: 0.2021 - val_recall: 0.2021
Epoch 3/30
22/22 - 0s - 6ms/step - accuracy: 0.4941 - loss: 0.6931 - precision: 0.4965 - recall: 0.2109 - val_accuracy: 0.7957 - val_loss: 0.6930 - val_precision: 0.7957 - val_recall: 0.7957
Epoch 4/30
22/22 - 0s - 6ms/step - accuracy: 0.4381 - loss: 0.6932 - precision: 0.4514 - recall: 0.3835 - val_accuracy: 0.7957 - val_loss: 0.6931 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 5/30
22/22 - 0s - 6ms/step - accuracy: 0.5000 - loss: 0.6932 - precision: 0.4954 - recall: 0.4720 - val_accuracy: 0.7957 - val_loss: 0.6919 - val_precision: 0.7957 - val_recall: 0.7957
Epoch 6/30
22/22 - 0s - 5ms/step - accuracy: 0

ValueError: A chosen class has no samples.

In [ ]:
import pandas as pd

def print_per_class_baseline_results(arrays, results_file):
    try:
        df = pd.read_json(results_file, lines=True)
    except Exception as e:
        print(f"Could not read results file: {e}")
        return

    baseline_df = df[df["run_type"] == "baseline"].copy()
    class_names = list(arrays.class_names)

    # Calculate the longest class name
    max_cls_len = max((len(str(cls)) for cls in class_names), default=0)
    col_width = max(max_cls_len + 2, 15)

    header = f"{'Target Class':<{col_width}} | {'Precision':>9} | {'Recall':>9} | {'Epochs':>6} | {'Timestamp'}"
    print(header)
    print("-" * len(header))

    for cls in class_names:
        cls_str = f"'{cls}'"
        matches = baseline_df[baseline_df["target_class"] == cls]
        
        row = matches.iloc[-1]
        prec = row.get("precision")
        rec = row.get("recall")
        epochs = row.get("epochs_trained")
        timestamp = row.get("timestamp")
        # Print the aligned single row
        print(f"{cls_str:<{col_width}} | {prec:>9.4f} | {rec:>9.4f} | {epochs:>6.0f} | {timestamp!s}")

print_per_class_baseline_results(arrays, RESULTS_FILE)

In [ ]:
scaling_rows = run_experiments(
    arrays=arrays,
    config=config,
    n_samples=N_SAMPLES,
    k_values=range(2, len(arrays.class_names)),
    run_baseline=False,
    run_scaling=True,
)
print(f"New scaling runs: {len(scaling_rows)}")
if scaling_rows:
    print("Last scaling row:")
    print(scaling_rows[-1])

In [ ]:
baseline_metrics, summary_df = summarize_results(RESULTS_FILE)
print(f"Baseline recall: {baseline_metrics.recall:.4f}")
print(f"Baseline precision: {baseline_metrics.precision:.4f}")
summary_df

In [ ]:
plot_summary(summary_df, baseline=baseline_metrics)